# Zimba Town Council — Group 45 Master Notebook

Consolidated cleaning summary for all three datasets collected by the group:

1. Council Administration
2. CDF Projects (+ source posts)
3. District Profile

For each dataset this notebook reports: raw shape, duplicate detection/removal, and numeric outlier detection (IQR method), correction, and removal — matching the Detect → Judge → Act workflow used in `scripts/clean_council_admin.py` and `scripts/clean_cdf.py`.

Detailed per-dataset cleaning steps and narrative live in `notebooks/cdf_cleaning.ipynb` and `notebooks/district_profile_cleaning.ipynb`; this notebook is the single top-level summary for submission.


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("../data")


def iqr_outlier_mask(frame, numeric_cols):
    mask = pd.Series(False, index=frame.index)
    for col in numeric_cols:
        q1, q3 = frame[col].quantile(0.25), frame[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        mask |= ((frame[col] < lower) | (frame[col] > upper)).fillna(False)
    return mask


def clip_outliers(frame, numeric_cols):
    frame = frame.copy()
    for col in numeric_cols:
        q1, q3 = frame[col].quantile(0.25), frame[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        frame[col] = frame[col].clip(lower, upper)
    return frame


def summarize(name, df, numeric_cols):
    print(f"=== {name} ===")
    print("Rows =", df.shape[0])
    print("Columns =", df.shape[1])

    before = len(df)
    deduped = df.drop_duplicates()
    print("After duplicate removal: Rows =", deduped.shape[0], "Columns =", deduped.shape[1])
    print("Duplicates removed =", before - len(deduped))

    if numeric_cols:
        mask = iqr_outlier_mask(deduped, numeric_cols)
        print("Outliers detected =", int(mask.sum()))
        corrected = clip_outliers(deduped, numeric_cols)
        print("After outlier correction: Rows =", corrected.shape[0], "Columns =", corrected.shape[1])
        removed = deduped.loc[~mask]
        print("After outlier removal: Rows =", removed.shape[0], "Columns =", removed.shape[1])
    else:
        print("Outliers detected = Not applicable (no numeric measurement columns)")
    print()
    return deduped


## 1. Council Administration Dataset

In [ ]:
admin_df = pd.read_csv(DATA / "db-unza26-csc4792-zimba_town_council_admin.csv", sep="|")
admin_df.head()


In [ ]:
# No numeric measurement columns (record_id is an identifier, not a measurement)
admin_clean = summarize("Council Administration", admin_df, numeric_cols=[])


## 2. CDF Projects Dataset

In [ ]:
cdf_projects_df = pd.read_csv(DATA / "db-unza26-csc4792-zimba_town_council_cdf_projects.csv", sep="|")
cdf_projects_df.head()


In [ ]:
cdf_clean = summarize("CDF Projects", cdf_projects_df, numeric_cols=["funding_amount_zmw"])


### 2b. CDF Source Posts

In [ ]:
cdf_posts_df = pd.read_csv(DATA / "cdf_source_posts.csv", sep="|")
cdf_posts_clean = summarize("CDF Source Posts", cdf_posts_df, numeric_cols=[])


## 3. District Profile Dataset

In [ ]:
district_df = pd.read_csv(DATA / "db-unza26-csc4792-zimba_town_council_district_profile.csv", sep="|")
district_df.head()


In [ ]:
district_numeric = ["population", "population_year", "population_male",
                    "population_female", "households", "area_km2"]
district_clean = summarize("District Profile", district_df, numeric_cols=district_numeric)


## Summary

| Dataset | Rows | Columns | Duplicates removed | Outliers detected |
|---|---|---|---|---|
| Council Administration | see output above | see output above | see output above | N/A |
| CDF Projects | see output above | see output above | see output above | see output above |
| CDF Source Posts | see output above | see output above | see output above | N/A |
| District Profile | see output above | see output above | see output above | see output above |

Run all cells above and fill in this table with the printed values for your team submission.
